In [ ]:
import xarray as xr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ERA5_total_prec = "/home/admin_climatecharted_com/data/ERA5_tp_hourly/ERA5_TP_ITA_2005-2014.grib"
ds = xr.open_dataset(ERA5_total_prec, engine="cfgrib")

print(ds)
print(ds.data_vars)
print(ds.attrs)
print(ds.tp.attrs)
print(ds.time.min().values, ds.time.max().values)

In [ ]:
ds.tp.isel(time=0).plot()

In [ ]:
tp_mm = ds.tp * 1000  # meters -> mm
max_rain = tp_mm.max(dim=["time", "step"])

plt.figure(figsize=(8,6))
max_rain.plot(
    cmap="Blues",        # color map
    cbar_kwargs={"label": "Max rainfall (mm)"}
)
plt.title("Maximum ERA5 Total Precipitation (2005-2014)")
plt.show()

In [ ]:
import xarray as xr
import geopandas as gpd
import matplotlib.pyplot as plt
import cartopy.crs as ccrs

# Load Italian shapefile
ita = gpd.read_file(r"/home/admin_climatecharted_com/data/it_4326/it_4326.shp")

# Convert ERA5 tp to mm and get max over time+step
tp_mm = ds.tp * 1000
max_rain = tp_mm.max(dim=["time", "step"])

# Plot with Cartopy
plt.figure(figsize=(10,10))
ax = plt.axes(projection=ccrs.PlateCarree())

# Plot ERA5 max rainfall
max_rain.plot(
    ax=ax,
    cmap="Blues",
    cbar_kwargs={"label": "Max rainfall (mm)"},
    transform=ccrs.PlateCarree()
)

# Overlay Italy shapefile
ita.boundary.plot(ax=ax, edgecolor='black', linewidth=1)

# Optional: coastlines and borders
ax.coastlines(resolution='10m')
ax.set_title("Maximum ERA5 Total Precipitation (2005-2014) over Italy")

plt.show()


In [ ]:
def load_era5_variable(name):
    """
    Loads a variable from its mapped GRIB file using cfgrib,
    and merges it with the matching 47N–48N file if it exists.
    """
    path = ERA5_VARIABLE_PATHS.get(name)
    if not path:
        raise ValueError(f"No GRIB file mapped for variable '{name}'")

    print(f"Loading {name} from {path}")
    
    return xr.open_dataset(
        path,
        engine="cfgrib",
        decode_timedelta=True
    )

def flatten_to_valid_time(var):
    """
    Flattens a variable with (time, step) dimensions into a single time axis (valid_time).
    """
    if "valid_time" not in var.coords:
        print(var.time.dtype, var.step.dtype)
        valid_time = var.time.values[:, None] + var.step.values[None, :]
        var = var.assign_coords(valid_time=(("time", "step"), valid_time))
    # else: print("valid_time is in var.coords")

    var = var.stack(datetime=("time", "step"))
    valid_times = var.valid_time.values.ravel()
    var = var.drop_vars(["time", "step", "valid_time"], errors="ignore")

    var = var.assign_coords(datetime=("datetime", pd.to_datetime(valid_times)))
    
    var = var.sortby("datetime")
    
    return var

def load_flatten_slice(da,
                       start,
                       end,
                       every_6h=False,
                       keep_hours=(0, 6, 12, 18)):
    """
    flatten time+step -> (optionally) keep 6h slots
    -> slice by datetime range.
    """
    
    if "step" not in da.dims:
        # print("Step not in dims")
        da = da.rename({"time": "datetime"})
        da = da.drop_vars(["step", "valid_time"], errors="ignore")
    else:  
        # print("Flattening to valid time...")
        da = flatten_to_valid_time(da)         
        # da = da * 1000  # meters → mm
        
    if every_6h:
        da = da.sel(datetime=da.datetime.dt.hour.isin(keep_hours))

    da = da.sel(datetime=slice(start, end))

    da = da.drop_vars(["number", "surface"], errors="ignore")
    
    return da.transpose("datetime", "latitude", "longitude")

In [ ]:
if time_interval == "2015-2025":
    start_time = "2015-01-01T00:00:00"
    end_time   = "2025-06-08T23:00:00"
elif time_interval == "2005-2014":
    start_time = "2005-01-01T00:00:00"
    end_time   = "2014-12-31T23:00:00"
elif time_interval == "1995-2004":
    start_time = "1995-01-01T00:00:00"
    end_time   = "2004-12-31T23:00:00"
elif time_interval == "1985-1994":
    start_time = "1985-01-01T00:00:00"
    end_time   = "1994-12-31T23:00:00"
elif time_interval == "1973-1984":
    start_time = "1973-01-01T00:00:00"
    end_time   = "1984-12-31T23:00:00"
else:
    raise ValueError(f"Unknown time interval: {time_interval}")

In [ ]:
def nan_stats(arr):
    return pd.Series({
        'total_cells'   : arr.size,
        'nan_cells'     : arr.isnull().sum().item(),
        'pct_nan'       : float(arr.isnull().mean().item())*100,
        'all_time_nan_cells': int((arr.isnull().all(dim='datetime')).sum()),
        'all_space_nan_times': int((arr.isnull().all(dim=['latitude','longitude'])).sum())
    })

In [ ]:
summary = pd.DataFrame({name: nan_stats(da) for name, da in {
    "tp": tp
}.items()}).T

print(summary)